In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import bootstrap
from tqdm.auto import tqdm


input_dir = Path("./baselines_mrr")

total_resamples = 10_000
resamples_per_chunk = 100
batch = 100

confidence_level = 0.95

rng = np.random.default_rng(42)

results = []

for path in sorted(input_dir.glob("*_example_mrr.csv")):
    model_name = path.name.replace("_example_mrr.csv", "")

    df = pd.read_csv(path)
    values = df["mrr"].to_numpy(dtype=float)

    observed_mrr = values.mean()

    bootstrap_result = None
    completed = 0

    pbar = tqdm(
        total=total_resamples,
        desc=f"Bootstrapping {model_name}",
        unit="resample",
    )

    while completed < total_resamples:
        n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

        bootstrap_result = bootstrap(
            data=(values,),
            statistic=np.mean,
            n_resamples=n_this_chunk,
            confidence_level=confidence_level,
            method="percentile",
            batch=batch,
            rng=rng,
            bootstrap_result=bootstrap_result,
        )

        completed += n_this_chunk
        pbar.update(n_this_chunk)

    pbar.close()

    ci = bootstrap_result.confidence_interval

    results.append(
        {
            "model": model_name,
            "n_examples": len(values),
            "observed_mrr": observed_mrr,
            "ci_low": ci.low,
            "ci_high": ci.high,
            "ci_width": ci.high - ci.low,
            "standard_error": bootstrap_result.standard_error,
            "confidence_level": confidence_level,
            "bootstrap_iterations": total_resamples,
        }
    )

bootstrap_results = (
    pd.DataFrame(results)
    .sort_values("observed_mrr", ascending=False)
    .reset_index(drop=True)
)

bootstrap_results

Bootstrapping bl_knn:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping bl_proxy:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedBase:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendeddouble:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedmlp:   0%|          | 0/10000 [00:00<?, ?resample/s]

...

Bootstrapping last_item:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping most_popular_past:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping original:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping random:   0%|          | 0/10000 [00:00<?, ?resample/s]

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,115701,0.086193,0.085120,0.087300,0.002180,0.000560,0.95,10000
1,extendedmlploss,115701,0.083633,0.082547,0.084697,0.002150,0.000544,0.95,10000
2,bl_proxy,115701,0.081746,0.080811,0.082669,0.001858,0.000475,0.95,10000
3,last_item,115701,0.064914,0.064115,0.065722,0.001607,0.000414,0.95,10000
4,extendedBase,115701,0.040199,0.039517,0.040889,0.001372,0.000353,0.95,10000
5,original,115701,0.036455,0.035834,0.037092,0.001257,0.000322,0.95,10000
6,extendeddouble,115701,0.033079,0.032441,0.033729,0.001288,0.000329,0.95,10000
7,most_popular_past,115701,0.026922,0.026289,0.027565,0.001276,0.000327,0.95,10000
8,bl_knn,115701,0.025385,0.024799,0.025971,0.001172,0.000303,0.95,10000
9,random,115701,0.001480,0.001349,0.001619,0.000270,0.000069,0.95,10000


In [2]:
bootstrap_results

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,115701,0.086193,0.085120,0.087300,0.002180,0.000560,0.95,10000
1,extendedmlploss,115701,0.083633,0.082547,0.084697,0.002150,0.000544,0.95,10000
2,bl_proxy,115701,0.081746,0.080811,0.082669,0.001858,0.000475,0.95,10000
3,last_item,115701,0.064914,0.064115,0.065722,0.001607,0.000414,0.95,10000
4,extendedBase,115701,0.040199,0.039517,0.040889,0.001372,0.000353,0.95,10000
5,original,115701,0.036455,0.035834,0.037092,0.001257,0.000322,0.95,10000
6,extendeddouble,115701,0.033079,0.032441,0.033729,0.001288,0.000329,0.95,10000
7,most_popular_past,115701,0.026922,0.026289,0.027565,0.001276,0.000327,0.95,10000
8,bl_knn,115701,0.025385,0.024799,0.025971,0.001172,0.000303,0.95,10000
9,random,115701,0.001480,0.001349,0.001619,0.000270,0.000069,0.95,10000


In [3]:
mlp_path = './baselines_mrr/extendedmlploss_example_mrr.csv'
bl_path = './baselines_mrr/bl_proxy_example_mrr.csv'

In [4]:
df1 = pd.read_csv(mlp_path)
bl = pd.read_csv(bl_path)

In [5]:
diff = pd.merge(df1, bl, on="example_index", how="inner", suffixes=("_mlp", "_bl"))

In [6]:
diff['diff'] = diff['mrr_mlp'] - diff['mrr_bl']

In [7]:

values = diff["diff"].to_numpy(dtype=float)

observed_diff = values.mean()

bootstrap_result = None
completed = 0

pbar = tqdm(
    total=total_resamples,
    unit="resample",
)

while completed < total_resamples:
    n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

    bootstrap_result = bootstrap(
        data=(values,),
        statistic=np.mean,
        n_resamples=n_this_chunk,
        confidence_level=confidence_level,
        method="percentile",
        batch=batch,
        rng=rng,
        bootstrap_result=bootstrap_result,
    )

    completed += n_this_chunk
    pbar.update(n_this_chunk)

pbar.close()

ci = bootstrap_result.confidence_interval

results.append(
    {
        ""
        "n_examples": len(values),
        "ci_low": ci.low,
        "ci_high": ci.high,
        "ci_width": ci.high - ci.low,
        "standard_error": bootstrap_result.standard_error,
        "confidence_level": confidence_level,
        "bootstrap_iterations": total_resamples,
    }
)

  0%|          | 0/10000 [00:00<?, ?resample/s]

In [8]:
ci = bootstrap_result.confidence_interval
ci

ConfidenceInterval(low=np.float64(0.0006328751401768316), high=np.float64(0.0031125310049464694))